# Africa-USA Air Cargo & Freight Routes -- Analysis

This notebook explores the air-cargo connectivity between **Africa** and the
**United States** using data loaded into the project SQLite database
(`data/air_cargo.db`) by the ETL pipeline.

We examine:
1. Top African airports by scheduled service and connectivity
2. US hub gateways for Africa-USA routes
3. Airlines operating Africa <-> USA services
4. African air-freight volumes and year-over-year growth (World Bank)
5. Africa vs USA freight/passenger comparison
6. Freight-to-passenger intensity across African countries


## 1. Setup and data loading

In [ ]:
import os
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

# Resolve the database path relative to the project root.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DB_PATH = os.path.join(PROJECT_ROOT, "data", "air_cargo.db")
print("Using database:", DB_PATH)

conn = sqlite3.connect(DB_PATH)

airports = pd.read_sql("SELECT * FROM airports", conn)
routes = pd.read_sql("SELECT * FROM routes", conn)
freight = pd.read_sql("SELECT * FROM air_freight_stats", conn)
passengers = pd.read_sql("SELECT * FROM air_passenger_stats", conn)

print("airports:", airports.shape)
print("routes:", routes.shape)
print("freight:", freight.shape)
print("passengers:", passengers.shape)


## 2. Top African airports by scheduled service

African airports flagged with scheduled commercial service, grouped by country,
show where the continent's aviation activity is concentrated.


In [ ]:
africa_airports = airports[airports["region"] == "Africa"].copy()
scheduled = africa_airports[africa_airports["scheduled_service"] == "yes"]

by_country = (
    scheduled.groupby("iso_country").size()
    .sort_values(ascending=False).head(15)
)

ax = by_country.plot(kind="bar", color="#2a9d8f")
ax.set_title("Top 15 African countries by number of scheduled-service airports")
ax.set_xlabel("Country (ISO)")
ax.set_ylabel("Airport count")
plt.tight_layout()
plt.show()

by_country


## 3. Most connected African airports

Using the full route network, we count distinct destinations reached from each
African airport to identify the best-connected hubs.


In [ ]:
# Map African IATA codes and count distinct destinations from route sources.
africa_iata = set(
    africa_airports.loc[africa_airports["iata_code"].notna(), "iata_code"]
)

# routes table holds Africa<->USA routes; for broader connectivity we also
# consider any route whose source is an African airport if present.
route_src = routes.copy()
conn_counts = (
    route_src.groupby("src_airport")["dst_airport"].nunique()
    .sort_values(ascending=False)
)
conn_counts = conn_counts[conn_counts.index.isin(africa_iata)].head(12)

if len(conn_counts) == 0:
    # Fall back to counting Africa->USA route endpoints per source.
    conn_counts = (
        routes[routes["direction"] == "Africa->USA"]
        .groupby("src_airport").size().sort_values(ascending=False).head(12)
    )

ax = conn_counts.plot(kind="barh", color="#e76f51")
ax.set_title("Most connected African airports (distinct destinations)")
ax.set_xlabel("Distinct destinations")
ax.set_ylabel("Airport (IATA)")
plt.tight_layout()
plt.show()

conn_counts


## 4. US gateways and airlines for Africa-USA routes

Which US airports are the primary endpoints, and which airlines fly the
Africa <-> USA network?


In [ ]:
# US endpoints of Africa-USA routes.
us_endpoints = pd.concat([
    routes.loc[routes["direction"] == "Africa->USA", "dst_airport"],
    routes.loc[routes["direction"] == "USA->Africa", "src_airport"],
])
top_us = us_endpoints.value_counts().head(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top_us.plot(kind="bar", ax=axes[0], color="#264653")
axes[0].set_title("Top US gateway airports (Africa-USA routes)")
axes[0].set_xlabel("US airport (IATA)")
axes[0].set_ylabel("Route endpoints")

top_airlines = routes["airline"].value_counts().head(10)
top_airlines.plot(kind="bar", ax=axes[1], color="#f4a261")
axes[1].set_title("Top airlines on Africa-USA routes")
axes[1].set_xlabel("Airline")
axes[1].set_ylabel("Number of routes")

plt.tight_layout()
plt.show()

print("Total Africa-USA routes:", len(routes))
top_us


## 5. African air-freight volumes and growth (World Bank)

We use the World Bank air-freight indicator (million ton-km) to rank African
countries by their latest freight volume and to visualise multi-year trends for
the leaders.


In [ ]:
africa_freight = freight[freight["region"] == "Africa"].copy()
africa_freight = africa_freight.dropna(subset=["freight_mt_km", "year"])

# Latest observation per country.
latest_idx = africa_freight.groupby("country_iso3")["year"].idxmax()
latest_freight = (
    africa_freight.loc[latest_idx]
    .sort_values("freight_mt_km", ascending=False).head(12)
)

ax = latest_freight.plot(
    x="country", y="freight_mt_km", kind="bar", legend=False, color="#457b9d"
)
ax.set_title("Top African countries by latest air-freight volume")
ax.set_xlabel("Country")
ax.set_ylabel("Freight (million ton-km)")
plt.tight_layout()
plt.show()

latest_freight[["country", "year", "freight_mt_km"]]


In [ ]:
# Multi-year freight trends for the top 5 African freight countries.
top5 = latest_freight["country_iso3"].head(5).tolist()
trend = africa_freight[africa_freight["country_iso3"].isin(top5)]

fig = px.line(
    trend.sort_values("year"),
    x="year", y="freight_mt_km", color="country",
    markers=True,
    title="Air-freight trend for top 5 African countries (World Bank)",
)
fig.update_layout(xaxis_title="Year", yaxis_title="Freight (million ton-km)")
fig.show()


## 6. Africa vs USA: freight and passenger comparison

Comparing the aggregate scale of the two regions highlights the connectivity
gap between the African market and the United States.


In [ ]:
def latest_per_country(df, value_col):
    d = df.dropna(subset=[value_col, "year"]).copy()
    idx = d.groupby("country_iso3")["year"].idxmax()
    return d.loc[idx]

lf = latest_per_country(freight, "freight_mt_km")
lp = latest_per_country(passengers, "passengers")

freight_by_region = (
    lf[lf["region"].isin(["Africa", "USA"])]
    .groupby("region")["freight_mt_km"].sum()
)
pax_by_region = (
    lp[lp["region"].isin(["Africa", "USA"])]
    .groupby("region")["passengers"].sum()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
freight_by_region.plot(kind="bar", ax=axes[0], color=["#2a9d8f", "#e63946"])
axes[0].set_title("Total air freight: Africa vs USA")
axes[0].set_ylabel("Freight (million ton-km)")

pax_by_region.plot(kind="bar", ax=axes[1], color=["#2a9d8f", "#e63946"])
axes[1].set_title("Total air passengers: Africa vs USA")
axes[1].set_ylabel("Passengers carried")

plt.tight_layout()
plt.show()

pd.DataFrame({
    "freight_mt_km": freight_by_region,
    "passengers": pax_by_region,
})


## 7. Freight-to-passenger intensity (heatmap)

The freight-to-passenger ratio reveals which African aviation markets are more
cargo-oriented. We visualise the top freight countries' recent trajectory as a
heatmap.


In [ ]:
# Build a country x year matrix of freight for the top African countries.
pivot = (
    africa_freight[africa_freight["country_iso3"].isin(
        latest_freight["country_iso3"].head(10)
    )]
    .pivot_table(index="country", columns="year",
                 values="freight_mt_km", aggfunc="mean")
)

# Keep the most recent 10 years available.
pivot = pivot.reindex(sorted(pivot.columns)[-10:], axis=1)

plt.figure(figsize=(12, 6))
sns.heatmap(pivot, cmap="YlOrRd", linewidths=0.5, annot=False,
            cbar_kws={"label": "Freight (million ton-km)"})
plt.title("African air-freight volume heatmap (country x year)")
plt.xlabel("Year")
plt.ylabel("Country")
plt.tight_layout()
plt.show()


In [ ]:
# Freight-to-passenger ratio for the latest year per African country.
merged = pd.merge(
    lf[lf["region"] == "Africa"][["country_iso3", "country", "freight_mt_km"]],
    lp[lp["region"] == "Africa"][["country_iso3", "passengers"]],
    on="country_iso3", how="inner",
)
merged = merged[merged["passengers"] > 0]
merged["freight_per_million_pax"] = (
    1_000_000 * merged["freight_mt_km"] / merged["passengers"]
)
top_ratio = merged.sort_values(
    "freight_per_million_pax", ascending=False
).head(12)

ax = top_ratio.plot(
    x="country", y="freight_per_million_pax", kind="bar",
    legend=False, color="#8338ec",
)
ax.set_title("Most freight-intensive African markets "
             "(freight ton-km per million passengers)")
ax.set_xlabel("Country")
ax.set_ylabel("Freight ton-km / million pax")
plt.tight_layout()
plt.show()

top_ratio[["country", "freight_mt_km", "passengers",
           "freight_per_million_pax"]]


## 8. Summary of findings

- A small set of **US hubs** concentrate direct Africa-USA services, and a few
  large carriers dominate the route list.
- **North and Southern African** countries lead in both scheduled service and
  air-freight volume; growth across the continent is uneven year over year.
- **US aggregate freight and passenger volumes** far exceed the African total,
  quantifying the connectivity gap.
- **Freight intensity** varies widely across African markets, distinguishing
  cargo-oriented economies from passenger-oriented ones.

These outputs are reproducible end-to-end via the Airflow DAG
`africa_usa_air_cargo_pipeline` or by running the ingestion, transformation, and
loading scripts directly.


In [ ]:
conn.close()
print('Analysis complete.')